# RRU7182M Antenna Mapping Analysis

Analyzes CSV output from `bench_scripts/mapping_logger.py` (which captures the ESP32's antenna-mapping firmware stream) to answer three questions:

1. **Where does each antenna actually detect a tag** — on the board, and how far off it?
2. **What RSSI threshold** (per antenna) would minimize false reads from bags/tags off the board while still reliably catching bags on it?
3. **What's the lowest RF power setting** that still reliably covers the board? Lower power both tightens the off-board false-read radius and meaningfully extends battery life for field deployment (current draw scales steeply with power).

See the testing protocol in `analysis/TESTING_STRATEGY.md` for the grid layout, label naming convention, power-level sweep procedure, and data collection procedure this notebook expects.

## 1. Configuration

Edit the board geometry and antenna positions below to match your actual physical layout before running the rest of the notebook — the defaults are placeholders.

In [1]:
import re
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

pd.set_option('display.max_rows', 50)

In [3]:
# ---- Board geometry, in inches. EDIT to match your actual board. ----
# Standard regulation cornhole board: 24" x 48", 6" hole centered 9" from the back edge.
BOARD_WIDTH = 24.0    # x extent of the playing surface
BOARD_LENGTH = 48.0   # y extent of the playing surface
HOLE_DIAMETER = 6.0
HOLE_CENTER_X = BOARD_WIDTH / 2
HOLE_CENTER_Y = BOARD_LENGTH - 9.0

# ---- Coordinate system ----
# Origin (0,0) = front-left corner of the playing surface (the edge closest to the
# thrower). +x = right, +y = away from the thrower (toward the hole/back edge).
# This must match the label convention used when collecting data -- see
# TESTING_STRATEGY.md. Points outside [0,BOARD_WIDTH] x [0,BOARD_LENGTH] are
# "off-board" by definition below.

# ---- Antenna physical positions (inches), same coordinate system as above.
# EDIT to match your actual antenna mounting layout. ----
ANTENNA_POSITIONS = {
    1: (-3, -3),                          # front-left
    2: (BOARD_WIDTH + 3, -3),              # front-right
    3: (BOARD_WIDTH + 3, BOARD_LENGTH + 3),# back-right
    4: (-3, BOARD_LENGTH + 3),             # back-left
}

# ---- RF power ----
# The reader's valid range is 0-30 (30 ~= 1W output, ~2W at the reader's own
# default of 33 which exceeds the documented range -- see main README). This is
# NOT a percentage scale, despite '10' sounding like one -- 10 is roughly a
# third of max power, not a tenth.
# Which power level to focus the single-power plots on (Sections 4-6 below).
# If your data only has one power level, this can just match it. If you've
# collected a power sweep, change this to inspect each level in turn, or see
# Section 7 for a plot across all levels at once.
POWER_TO_PLOT = 10

# ---- Data files ----
# By default, loads every mapping_*.csv in the current directory (mapping_logger.py's
# default naming). Point this at specific files/paths if your session files live elsewhere.
# Since rf_power is now recorded per-row in the CSV itself, sessions collected at
# different power levels can be loaded together and compared -- no need to keep
# them in separate notebook runs.
CSV_FILES = sorted(glob.glob('mapping_*.csv'))
print(f'Found {len(CSV_FILES)} file(s):')
for f in CSV_FILES:
    print(' ', f)

Found 1 file(s):
  mapping_20260816_124408.csv


## 2. Load and parse

Expects the label convention `x<X>_y<Y>` (inches, matching the config above) — e.g. `x12_y24`. Rows with unparseable labels (including the default `unlabeled`) are kept but will show up as `NaN` position and get dropped from the position-based plots later; they're still usable for basic RSSI sanity checks.

**Older CSV files collected before `rf_power` was added to the firmware** won't have that column — these get flagged rather than silently mixed in with an assumed value, since guessing wrong here would quietly corrupt any power-comparison analysis.

In [5]:
LABEL_RE = re.compile(r'^x(-?\d+\.?\d*)_y(-?\d+\.?\d*)$')

def parse_label(label):
    m = LABEL_RE.match(str(label))
    if not m:
        return (np.nan, np.nan)
    return (float(m.group(1)), float(m.group(2)))

if not CSV_FILES:
    raise FileNotFoundError(
        'No mapping CSV files found. Check CSV_FILES glob pattern in the config cell, '
        'or run mapping_logger.py first to generate one.'
    )

frames = []
old_format_files = []
for path in CSV_FILES:
    df = pd.read_csv(path, dtype=str)
    if 'rf_power' not in df.columns:
        old_format_files.append(path)
        continue
    df['source_file'] = path
    frames.append(df)

if old_format_files:
    print('SKIPPED (no rf_power column -- collected before this notebook version):')
    for f in old_format_files:
        print(' ', f)
    print('Re-run these sessions with the current firmware if you need them included.\n')

if not frames:
    raise ValueError('No usable CSV files with an rf_power column were found.')

raw = pd.concat(frames, ignore_index=True)
raw['antenna'] = raw['antenna'].astype(int)
raw['rf_power'] = pd.to_numeric(raw['rf_power'], errors='coerce').astype('Int64')
raw['rssi'] = pd.to_numeric(raw['rssi'], errors='coerce')  # NaN for non-detection rows
raw['detected'] = raw['epc'].apply(
    lambda e: isinstance(e, str) and re.fullmatch(r'[0-9A-Fa-f]+', e) is not None
)
raw[['x_in', 'y_in']] = raw['label'].apply(lambda l: pd.Series(parse_label(l)))
raw['on_board'] = raw['x_in'].between(0, BOARD_WIDTH) & raw['y_in'].between(0, BOARD_LENGTH)

print(f"{len(raw)} total rows")
print(f"{raw['detected'].sum()} tag detections ({raw['detected'].mean():.1%} of rows)")
print(f"{raw['label'].nunique()} distinct labels")
print(f"Power levels present: {sorted(raw['rf_power'].dropna().unique().tolist())}")
print(f"{raw['x_in'].isna().sum()} rows with unparseable labels (excluded from position plots)")
raw.head(10)

SKIPPED (no rf_power column -- collected before this notebook version):
  mapping_20260816_124408.csv
Re-run these sessions with the current firmware if you need them included.



ValueError: No usable CSV files with an rf_power column were found.

### Non-detection status rows

Besides `NONE` (no tag seen), the firmware also emits `TIMEOUT`, `BAD_FRAME`, `ANTENNA_ERROR`, or `STATUS_0x..` in the `epc` column when something went wrong at the protocol level rather than simply "no tag in range". Worth a quick look — a cluster of these at a particular antenna or time window points at a comms/hardware issue, not a mapping result.

In [ ]:
status_counts = raw.loc[~raw['detected'], 'epc'].value_counts()
status_counts

## 3. Per-position, per-antenna, per-power summary

Aggregates every cycle at each `(label, antenna, rf_power)` combination. `detection_rate` matters as much as `rssi_mean` here — EPC Gen2's anti-collision slotting means even a tag well within range won't answer every single query, so a low detection rate at a position that's otherwise inside the board is a signal worth a second look, not necessarily a dead zone.

In [ ]:
summary = (
    raw.groupby(['label', 'x_in', 'y_in', 'antenna', 'rf_power', 'on_board'], dropna=False)
    .agg(
        cycles=('detected', 'size'),
        detections=('detected', 'sum'),
        rssi_mean=('rssi', 'mean'),
        rssi_std=('rssi', 'std'),
        rssi_min=('rssi', 'min'),
        rssi_max=('rssi', 'max'),
    )
    .reset_index()
)
summary['detection_rate'] = summary['detections'] / summary['cycles']
summary.sort_values(['rf_power', 'antenna', 'y_in', 'x_in']).head(20)

## 4. Heatmaps per antenna (at `POWER_TO_PLOT`)

Two views per antenna: mean RSSI where detected, and detection rate (0-1) across all cycles, filtered to the power level set in the config cell. The plot grid sizes itself to whichever antennas are actually present in the data — a single panel if you're running single-antenna mode, up to the full 2x2 grid once all 4 are back in use, no notebook edits needed either way. The board outline, hole, and antenna position are drawn for reference — check these render in roughly the right place for your layout before trusting the rest.

In [ ]:
def plot_board_outline(ax):
    ax.add_patch(patches.Rectangle((0, 0), BOARD_WIDTH, BOARD_LENGTH,
                                    fill=False, edgecolor='black', linewidth=2, zorder=5))
    ax.add_patch(patches.Circle((HOLE_CENTER_X, HOLE_CENTER_Y), HOLE_DIAMETER / 2,
                                 fill=False, edgecolor='black', linewidth=1,
                                 linestyle='--', zorder=5))

def plot_metric_heatmap(ax, data_at_power, metric_col, antenna, title,
                         cmap='viridis', vmin=None, vmax=None):
    data = data_at_power[(data_at_power['antenna'] == antenna)].dropna(subset=['x_in', 'y_in'])
    sc = ax.scatter(data['x_in'], data['y_in'], c=data[metric_col], s=260,
                     cmap=cmap, vmin=vmin, vmax=vmax, edgecolors='k', linewidths=0.6, zorder=3)
    pos = ANTENNA_POSITIONS.get(antenna)
    if pos:
        ax.plot(*pos, marker='^', color='red', markersize=14, zorder=6,
                label=f'Antenna {antenna}')
    plot_board_outline(ax)
    ax.set_title(title)
    ax.set_xlabel('x (in)')
    ax.set_ylabel('y (in)')
    ax.set_aspect('equal')
    ax.legend(loc='upper right', fontsize=8)
    return sc

In [ ]:
def subplot_grid_shape(n):
    """Rows/cols for n antenna subplots, sized exactly to n (no blank
    leftover panels) -- 1=1x1, 2=1x2, 3=1x3, 4=2x2."""
    return {1: (1, 1), 2: (1, 2), 3: (1, 3), 4: (2, 2)}.get(n, (1, n))

In [ ]:
summary_at_power = summary[summary['rf_power'] == POWER_TO_PLOT]
if summary_at_power.empty:
    print(f'No data at rf_power={POWER_TO_PLOT}. Available levels: '
          f"{sorted(summary['rf_power'].dropna().unique().tolist())}")
else:
    present_antennas = sorted(summary_at_power['antenna'].dropna().unique().tolist())
    rows, cols = subplot_grid_shape(len(present_antennas))
    fig, axes = plt.subplots(rows, cols, figsize=(7 * cols, 7.5 * rows), squeeze=False)
    axes_flat = axes.flatten()

    rssi_vals = summary_at_power.loc[summary_at_power['rssi_mean'].notna(), 'rssi_mean']
    vmin, vmax = (rssi_vals.min(), rssi_vals.max()) if len(rssi_vals) else (None, None)

    sc = None
    for ant, ax in zip(present_antennas, axes_flat):
        sc = plot_metric_heatmap(ax, summary_at_power, 'rssi_mean', ant,
                                  f'Antenna {ant} — Mean RSSI', vmin=vmin, vmax=vmax)

    if sc is not None:
        fig.colorbar(sc, ax=list(axes_flat), shrink=0.6, label='Mean RSSI')
    if len(present_antennas) < 4:
        print(f'Note: only {len(present_antennas)} antenna(s) in this data '
              f'({present_antennas}) -- plotting those only.')
    fig.suptitle(f'RSSI by position, per antenna (rf_power={POWER_TO_PLOT})', fontsize=16)
    plt.savefig(f'rssi_heatmaps_power{POWER_TO_PLOT}.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
if not summary_at_power.empty:
    fig, axes = plt.subplots(rows, cols, figsize=(7 * cols, 7.5 * rows), squeeze=False)
    axes_flat = axes.flatten()

    sc = None
    for ant, ax in zip(present_antennas, axes_flat):
        sc = plot_metric_heatmap(ax, summary_at_power, 'detection_rate', ant,
                                  f'Antenna {ant} — Detection rate', cmap='RdYlGn', vmin=0, vmax=1)

    if sc is not None:
        fig.colorbar(sc, ax=list(axes_flat), shrink=0.6, label='Detection rate (0-1)')
    fig.suptitle(f'Detection rate by position, per antenna (rf_power={POWER_TO_PLOT})', fontsize=16)
    plt.savefig(f'detection_rate_heatmaps_power{POWER_TO_PLOT}.png', dpi=150, bbox_inches='tight')
    plt.show()

## 5. On-board vs. off-board RSSI separation (at `POWER_TO_PLOT`)

The core question for tuning: is there a clean RSSI gap between on-board and off-board reads, or do they overlap? Pooled across all 4 antennas first, then broken out per-antenna below since antennas can differ quite a bit. All filtered to the power level set in the config cell.

In [ ]:
raw_at_power = raw[raw['rf_power'] == POWER_TO_PLOT]
data_on = raw_at_power.loc[raw_at_power['on_board'] & raw_at_power['detected'], 'rssi'].dropna()
data_off = raw_at_power.loc[(~raw_at_power['on_board']) & raw_at_power['detected'], 'rssi'].dropna()

fig, ax = plt.subplots(figsize=(9, 5))
bins = 20
if len(data_on):
    ax.hist(data_on, bins=bins, alpha=0.6, label=f'On-board (n={len(data_on)})', color='green')
if len(data_off):
    ax.hist(data_off, bins=bins, alpha=0.6, label=f'Off-board (n={len(data_off)})', color='red')
ax.set_xlabel('RSSI')
ax.set_ylabel('Count')
ax.set_title(f'RSSI distribution: on-board vs. off-board (rf_power={POWER_TO_PLOT}, all antennas)')
ax.legend()
plt.show()

if len(data_on) and len(data_off):
    on_p05, on_median = data_on.quantile(0.05), data_on.median()
    off_p95, off_median = data_off.quantile(0.95), data_off.median()
    candidate_threshold = (on_p05 + off_p95) / 2
    print(f'On-board  RSSI: min={data_on.min():.0f}  5th pct={on_p05:.1f}  median={on_median:.1f}')
    print(f'Off-board RSSI: max={data_off.max():.0f}  95th pct={off_p95:.1f}  median={off_median:.1f}')
    print(f'\nCandidate threshold at rf_power={POWER_TO_PLOT} (rough starting point, NOT a '
          f'rigorous cutoff): {candidate_threshold:.1f}')
    if on_p05 < off_p95:
        print('\nWARNING: on-board 5th-percentile RSSI is BELOW off-board 95th-percentile --')
        print('the distributions overlap at these tails. A single global threshold likely')
        print('cannot cleanly separate on/off-board here. Check per-antenna thresholds below --')
        print('one antenna dragging the pooled numbers down is a common cause. If it persists')
        print('per-antenna too, this power level may simply be too hot for clean separation --')
        print('see Section 7 for whether a lower power level does better.')
else:
    print(f'Not enough on-board and/or off-board detections at rf_power={POWER_TO_PLOT} to assess '
          'separation yet.')

### Per-antenna thresholds (at `POWER_TO_PLOT`)


In [ ]:
threshold_rows = []
for ant in [1, 2, 3, 4]:
    ant_on = raw_at_power.loc[raw_at_power['on_board'] & raw_at_power['detected'] &
                               (raw_at_power['antenna'] == ant), 'rssi'].dropna()
    ant_off = raw_at_power.loc[(~raw_at_power['on_board']) & raw_at_power['detected'] &
                                (raw_at_power['antenna'] == ant), 'rssi'].dropna()
    row = {'antenna': ant, 'rf_power': POWER_TO_PLOT, 'n_on': len(ant_on), 'n_off': len(ant_off)}
    if len(ant_on) and len(ant_off):
        on_p05 = ant_on.quantile(0.05)
        off_p95 = ant_off.quantile(0.95)
        row['on_p05'] = round(on_p05, 1)
        row['off_p95'] = round(off_p95, 1)
        row['candidate_threshold'] = round((on_p05 + off_p95) / 2, 1)
        row['clean_separation'] = bool(on_p05 >= off_p95)
    threshold_rows.append(row)

threshold_df = pd.DataFrame(threshold_rows)
threshold_df

## 6. Effective coverage at a chosen threshold (at `POWER_TO_PLOT`)

Pick a threshold (start from the per-antenna table above, or the global candidate) and see which grid points it would actually flag as "tag present" — a bag counts as detected if *any* antenna sees it above the threshold. Compare the shaded region against the true board outline to spot false-positive bleed (shaded outside the box) or blind spots (unshaded points inside it).

In [ ]:
# EDIT this after reviewing the tables above -- start with the global candidate
# or the lowest per-antenna candidate_threshold, then adjust by eye.
THRESHOLD = candidate_threshold if 'candidate_threshold' in dir() else -60

coverage = (
    summary_at_power.dropna(subset=['x_in', 'y_in'])
    .groupby(['label', 'x_in', 'y_in', 'on_board'])['rssi_max']
    .max()
    .reset_index()
)
coverage['flagged'] = coverage['rssi_max'] >= THRESHOLD

fig, ax = plt.subplots(figsize=(8, 12))
flagged = coverage[coverage['flagged']]
unflagged = coverage[~coverage['flagged']]
ax.scatter(flagged['x_in'], flagged['y_in'], c='green', s=200, label='Flagged as detected',
           edgecolors='k', zorder=3)
ax.scatter(unflagged['x_in'], unflagged['y_in'], c='lightgray', s=200, label='Not flagged',
           edgecolors='k', zorder=3)
plot_board_outline(ax)
ax.set_title(f'Effective coverage at threshold={THRESHOLD:.1f}, rf_power={POWER_TO_PLOT}')
ax.set_xlabel('x (in)')
ax.set_ylabel('y (in)')
ax.set_aspect('equal')
ax.legend(loc='upper right')
plt.savefig(f'effective_coverage_power{POWER_TO_PLOT}.png', dpi=150, bbox_inches='tight')
plt.show()

false_positives = coverage[coverage['flagged'] & ~coverage['on_board']]
false_negatives = coverage[~coverage['flagged'] & coverage['on_board']]
print(f'{len(false_positives)} off-board position(s) flagged as detected at this threshold/power')
print(f'{len(false_negatives)} on-board position(s) NOT flagged at this threshold/power')

## 7. Coverage vs. power level

If your data includes more than one `rf_power` level (run separate mapping sessions at different power settings — `MAPPING_RF_POWER` in `main.cpp` — then load all the CSVs together here), this is the section that actually answers "what's the lowest power that still works": on-board detection rate and off-board false-read rate, plotted against power. The power where the two lines start to converge (falling on-board coverage, but off-board bleed not dropping much further) is the practical floor — going lower than that starts losing real coverage without buying much additional off-board safety margin.

In [ ]:
power_levels = sorted(raw['rf_power'].dropna().unique().tolist())

if len(power_levels) < 2:
    print(f'Only one power level in the data ({power_levels}). Collect at least one more '
          'session at a different rf_power to use this section -- see "Antenna mapping mode" '
          'in the main README for how MAPPING_RF_POWER is set.')
else:
    rows = []
    for p in power_levels:
        d = raw[raw['rf_power'] == p]
        on_rate = d.loc[d['on_board'], 'detected'].mean() if d['on_board'].any() else np.nan
        off_rate = d.loc[~d['on_board'], 'detected'].mean() if (~d['on_board']).any() else np.nan
        rows.append({'rf_power': p, 'on_board_detection_rate': on_rate,
                     'off_board_detection_rate': off_rate})
    power_summary = pd.DataFrame(rows)

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(power_summary['rf_power'], power_summary['on_board_detection_rate'],
            marker='o', color='green', label='On-board detection rate')
    ax.plot(power_summary['rf_power'], power_summary['off_board_detection_rate'],
            marker='o', color='red', label='Off-board detection rate (false-read risk)')
    ax.set_xlabel('RF power (0-30 scale)')
    ax.set_ylabel('Detection rate')
    ax.set_ylim(-0.05, 1.05)
    ax.set_title('Coverage vs. RF power')
    ax.legend()
    ax.grid(alpha=0.3)
    plt.savefig('coverage_vs_power.png', dpi=150, bbox_inches='tight')
    plt.show()

    power_summary

## 8. Export

Saves the aggregated summary table and the threshold recommendations for reference outside this notebook. Both are power-aware — `rf_power` is a column in each, not something you have to remember separately. Figures were already saved as PNGs in the cells above.

In [ ]:
summary.to_csv('mapping_summary.csv', index=False)
threshold_df.to_csv('threshold_recommendations.csv', index=False)
print('Wrote mapping_summary.csv and threshold_recommendations.csv')